In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/temperature.csv", skiprows=8, skipinitialspace=True)

df["time"] = pd.to_datetime(df["time"])
df["temperature_c"] = df["mean_M2SDNXSLV_5_12_4_T2MMEAN"] - 273.15

df = df.sort_values("time").reset_index(drop=True)

print(df.head())
print(df.shape)


        time  mean_M2SDNXSLV_5_12_4_T2MMEAN  temperature_c
0 2000-01-01                     297.731354      24.581354
1 2000-01-02                     298.095612      24.945612
2 2000-01-03                     298.583588      25.433588
3 2000-01-04                     298.452271      25.302271
4 2000-01-05                     298.406769      25.256769
(9497, 3)


In [3]:
df["temp_lag1"] = df["temperature_c"].shift(1)
df["temp_lag2"] = df["temperature_c"].shift(2)
df["temp_lag7"] = df["temperature_c"].shift(7)
df["temp_rolling7"] = df["temperature_c"].shift(1).rolling(7).mean()
df["month"] = df["time"].dt.month

df = df.dropna().reset_index(drop=True)

features = [
    "temp_lag1",
    "temp_lag2",
    "temp_lag7",
    "temp_rolling7",
    "month"
]

target = "temperature_c"

print(df[["time"] + features + [target]].head())
print("Total observations:", len(df))


        time  temp_lag1  temp_lag2  temp_lag7  temp_rolling7  month  \
0 2000-01-08  23.739160  24.472192  24.581354      24.818707      1   
1 2000-01-09  24.312524  23.739160  24.945612      24.780302      1   
2 2000-01-10  24.909265  24.312524  25.433588      24.775110      1   
3 2000-01-11  24.930444  24.909265  25.302271      24.703232      1   
4 2000-01-12  24.916986  24.930444  25.256769      24.648191      1   

   temperature_c  
0      24.312524  
1      24.909265  
2      24.930444  
3      24.916986  
4      24.666040  
Total observations: 9490


In [4]:
train = df[df["time"].dt.year <= 2020].copy()
test = df[df["time"].dt.year >= 2021].copy()

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print("Training period:", train["time"].min(), "to", train["time"].max())
print("Testing period:", test["time"].min(), "to", test["time"].max())


Training observations: 7664
Testing observations: 1826
Training period: 2000-01-08 00:00:00 to 2020-12-31 00:00:00
Testing period: 2021-01-01 00:00:00 to 2025-12-31 00:00:00


In [6]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest training completed!")
print("Training score:", rf_model.score(X_train, y_train))
print("Testing score:", rf_model.score(X_test, y_test))


Random Forest training completed!
Training score: 0.7623549944525336
Testing score: 0.6111083590223025


In [7]:
y_pred = rf_model.predict(X_test)

results = test[["time", "temperature_c"]].copy()
results["predicted_temperature"] = y_pred

print(results.head(10))


           time  temperature_c  predicted_temperature
7664 2021-01-01      25.276361              25.339102
7665 2021-01-02      25.850244              25.326369
7666 2021-01-03      25.279077              25.625117
7667 2021-01-04      25.223016              25.404516
7668 2021-01-05      25.110010              25.246934
7669 2021-01-06      25.590509              25.208779
7670 2021-01-07      25.388666              25.403922
7671 2021-01-08      25.757318              25.363738
7672 2021-01-09      25.831415              25.622971
7673 2021-01-10      25.776697              25.639318


In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rf_mae = mean_absolute_error(y_test, y_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
rf_r2 = r2_score(y_test, y_pred)

print("Random Forest Results")
print(f"MAE: {rf_mae:.4f} °C")
print(f"RMSE: {rf_rmse:.4f} °C")
print(f"R²: {rf_r2:.4f}")


Random Forest Results
MAE: 0.3382 °C
RMSE: 0.4239 °C
R²: 0.6111


In [9]:
baseline_pred = X_test["temp_lag1"]

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("Baseline Results")
print(f"MAE: {baseline_mae:.4f} °C")
print(f"RMSE: {baseline_rmse:.4f} °C")
print(f"R²: {baseline_r2:.4f}")


Baseline Results
MAE: 0.3551 °C
RMSE: 0.4511 °C
R²: 0.5596


In [10]:
comparison = pd.DataFrame({
    "Model": ["Random Forest", "Persistence Baseline"],
    "MAE": [rf_mae, baseline_mae],
    "RMSE": [rf_rmse, baseline_rmse],
    "R²": [rf_r2, baseline_r2]
})

print(comparison.to_string(index=False))


               Model      MAE     RMSE       R²
       Random Forest 0.338234 0.423888 0.611108
Persistence Baseline 0.355127 0.451112 0.559552


In [11]:
import joblib
from pathlib import Path

Path("../models").mkdir(exist_ok=True)

joblib.dump(
    {"model": rf_model, "features": features},
    "../models/temperature_rf.joblib"
)

print("Model saved successfully!")


Model saved successfully!
